In [1]:
import pandas as pd
import polars as pl
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,f1_score

In [2]:
df = pl.read_parquet("../data/03_processed/baseline_features_train.parquet").to_pandas()
df = df.dropna(subset='IncidentGrade')
label_map = {
    "FalsePositive" : 0,
    'BenignPositive' : 1,
    'TruePositive' : 2
}
df['target'] = df['IncidentGrade'].map(label_map)


X = df.drop(columns=['IncidentGrade','start_time','end_time','target'])
y = df['target']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_test.shape}")
print(f"Target distribution:\n{y_train.value_counts(normalize=True) * 100}")

Training set shape: (358277, 33)
Validation set shape: (89570, 33)
Target distribution:
target
1    48.575823
0    30.119433
2    21.304745
Name: proportion, dtype: float64


In [4]:
rf_baseline = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
print("Training the Random Forest Model ...")
rf_baseline.fit(X_train,y_train)

print("Predicting on validation set ...")
y_pred = rf_baseline.predict(X_test)

print("\n--- Classification Report ---")
target_labels = ['False Positive (0)','Benign Positive (1)','True Positive (2)']
print(classification_report(y_test,y_pred,target_names=target_labels))

macro_f1 = f1_score(y_test, y_pred, average='macro')
print(f"\nCRITICAL BENCHMARK -> Baseline Macro-F1 Score: {macro_f1:.4f}")

Training the Random Forest Model ...
Predicting on validation set ...

--- Classification Report ---
                     precision    recall  f1-score   support

 False Positive (0)       0.86      0.67      0.75     26978
Benign Positive (1)       0.70      0.95      0.80     43509
  True Positive (2)       0.85      0.44      0.58     19083

           accuracy                           0.75     89570
          macro avg       0.80      0.68      0.71     89570
       weighted avg       0.78      0.75      0.74     89570


CRITICAL BENCHMARK -> Baseline Macro-F1 Score: 0.7114
